In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Deep Learning - K-Nearest Neighbors (KNN) Classification
# Camera vs Flamingo vs Pizza Image Classification
# Date: 31.10.2025

import cv2
import numpy as np
import os
import glob
from scipy import stats
from scipy.stats import skew
from scipy.fftpack import fft2
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:

# ------Feature extraction-----


def extract_features(image_path):


    epsilon = 1e-6  # to prevent division by zero

    # GROUP A: GRADIENT FEATURES (First 8 Attributes)

    # 1. Read the image in grayscale and resize to 128x128
    img_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise ValueError(f"Image could not be read: {image_path}")

    img_gray = cv2.resize(img_gray, (128, 128))

    # 2. Compute gradient map using Sobel filters
    sobelx = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=5)
    sobely = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=5)
    gradient = np.abs(sobelx) + np.abs(sobely)

    # 3. Flatten gradient image into a 1x16384 vector
    grad_flat = gradient.flatten()  # 128 * 128 = 16384

    # 4. Compute the first 8 features

    # Feature 1: Mean
    mean_val = np.mean(grad_flat)

    # Feature 2: Entropy
    hist, _ = np.histogram(grad_flat, bins=256, range=(0, grad_flat.max() + 1))
    hist_norm = hist / (hist.sum() + epsilon)
    hist_norm = hist_norm[hist_norm > 0]  # remove zeros to avoid log(0)
    entropy_val = -np.sum(hist_norm * np.log2(hist_norm + epsilon))

    # Feature 3: Standard Deviation
    std_dev_val = np.std(grad_flat)

    # Feature 4: Skewness
    skewness_val = skew(grad_flat)
    if np.isnan(skewness_val) or np.isinf(skewness_val):
        skewness_val = 0.0

    # Feature 5: Moment
    moment_val = np.var(grad_flat)

    # Feature 6: Mean FFT
    fft_val = np.abs(fft2(gradient))
    mean_fft_val = np.log(np.mean(fft_val) + 1)

    # Feature 7: Percentile
    percentile_val = np.percentile(grad_flat, 50)

    # Feature 8: Median
    median_val = np.median(grad_flat)

    # GROUP B: COLOR INDEX FEATURES (Last 5 Attributes)

    # 1. Read the image in color mode
    img_color = cv2.imread(image_path)
    if img_color is None:
        raise ValueError(f"Color image could not be read: {image_path}")

    img_color = cv2.resize(img_color, (128, 128))

    # 2. Compute the mean of R, G, and B channels
    B_mean, G_mean, R_mean = np.mean(img_color, axis=(0, 1))

    # 3. Compute the last 5 color features

    # Feature 9: Brightness Index
    brightness_idx = (R_mean**2 + G_mean**2 + B_mean**2) / 3

    # Feature 10: Saturation Index
    saturation_idx = (R_mean - B_mean) / (R_mean + B_mean + epsilon)

    # Feature 11: Hue Index
    hue_idx = (2 * R_mean - G_mean - B_mean) / (G_mean - B_mean + epsilon)
    hue_idx = np.clip(hue_idx, -100, 100)

    # Feature 12: Coloration Index
    coloration_idx = (R_mean - G_mean) / (R_mean + G_mean + epsilon)

    # Feature 13: Redness Index
    redness_idx = (R_mean**2) / (B_mean * (G_mean**3) + epsilon)
    redness_idx = np.clip(redness_idx, 0, 100)

    # ------Combine the all features-----

    features = np.array([
        mean_val,
        entropy_val,
        std_dev_val,
        skewness_val,
        moment_val,
        mean_fft_val,
        percentile_val,
        median_val,
        brightness_idx,
        saturation_idx,
        hue_idx,
        coloration_idx,
        redness_idx
    ])

    # Clean NaN/Inf values
    features = np.nan_to_num(features, nan=0.0, posinf=1e5, neginf=-1e5)

    # Use float64 for precision
    return features.astype('float64')

print("extract_features function defined.")


# -------Data Loading Function-------


def load_train_data(base_train_path):


    # Define class labels
    classes = {'camera': 1, 'flamingo': 2, 'pizza': 3}

    x_train = []
    y_train = []

    print("Processing training data")

    for class_name, label in classes.items():
        # Find all .jpg images in the class folder
        class_path = os.path.join(base_train_path, class_name)
        image_paths = glob.glob(os.path.join(class_path, '*.jpg'))

        # Check if the folder exists
        if not os.path.exists(class_path):
            print(f"WARNING: Folder not found: {class_path}. Skipping this class.")
            continue

        print(f"-> Processing {class_name} ({label}) class: {len(image_paths)} images")

        for img_path in image_paths:
            try:
                # Extract 13 features from each image
                features = extract_features(img_path)

                # Append to lists
                x_train.append(features)
                y_train.append(label)
            except Exception as e:
                print(f"Error! Could not process {img_path}: {e}")

    # Convert Python lists to NumPy arrays
    x_train = np.array(x_train)
    y_train = np.array(y_train)

    print("\nTraining data prepared.")
    print(f"x_train shape: {x_train.shape}")
    print(f"y_train shape: {y_train.shape}")

    return x_train, y_train


# ----KNN FUNCTION-----


def KNN(x_train, y_train, sample_test, k):

    # 1. Compute Euclidean distance between test sample and all training samples
    distances = np.sqrt(np.sum((x_train - sample_test)**2, axis=1))

    # 2. Sort distances and get indices of the k nearest samples
    k_nearest_indices = np.argsort(distances)[:k]

    # 3. Retrieve labels of the k nearest neighbors
    k_nearest_labels = y_train[k_nearest_indices]

    # 4. Get the most frequent label
    try:
        prediction = stats.mode(k_nearest_labels, keepdims=True).mode[0]
    except:
        prediction = stats.mode(k_nearest_labels)[0]

    # 5. Return the predicted label
    return int(prediction)

print("KNN function defined.")


# MAIN EXECUTION

def main():
    """
    Main execution function:
    1. Load training data
    2. Apply normalization for the accuracy
    3. Load test data and make predictions
    4. Calculate accuracy
    """

    # File paths
    base_train_path = '/content/drive/MyDrive/CaltechTinyThreeClass/train/'
    base_test_path = '/content/drive/MyDrive/CaltechTinyThreeClass/test/'

    # -------Step 1: Load Training Data--------
    print("=" * 60)
    print("STEP 1: TRAIN DATA LOADING")
    print("=" * 60)

    x_train, y_train = load_train_data(base_train_path)

    # -------Step 2 : Normalization-----
    print("\n" + "=" * 60)
    print("STEP 2: FEATURE NORMALIZATION")
    print("=" * 60)

    # Compute mean and std from training data only
    train_mean = np.mean(x_train, axis=0)
    train_std = np.std(x_train, axis=0)

    # Normalize training data
    x_train_normalized = (x_train - train_mean) / (train_std + 1e-6)

    print("Normalization completed.")
    print(f"Train mean: {train_mean[:3]} (first 3 features)")
    print(f"Train std: {train_std[:3]} (first 3 features)")
    print(f"Normalized data mean: {np.mean(x_train_normalized):.6f}")
    print(f"Normalized data std: {np.std(x_train_normalized):.6f}")

    # -----Step 3 : Testing-----
    print("\n" + "=" * 60)
    print("STEP 3: TESTING WITH KNN (k=5)")
    print("=" * 60)

    label_to_name = {1: 'camera', 2: 'flamingo', 3: 'pizza'}
    classes = {'camera': 1, 'flamingo': 2, 'pizza': 3}

    k_value = 5  # As specified in the assignment

    print(f"\nk value: {k_value}")
    print("Testing started")
    print("-" * 60)

    correct_predictions = 0
    total_predictions = 0

    class_correct = {1: 0, 2: 0, 3: 0}
    class_total = {1: 0, 2: 0, 3: 0}

    # Iterate through test folders
    for class_name, label in classes.items():
        class_path = os.path.join(base_test_path, class_name)
        image_paths = glob.glob(os.path.join(class_path, '*.jpg'))

        print(f"\nTesting {class_name} images")

        for img_path in image_paths:
            true_label = label
            sample_test_features = extract_features(img_path)

            # Normalize test sample using training statistics
            sample_normalized = (sample_test_features - train_mean) / (train_std + 1e-6)

            # Predict using KNN
            predicted_label = KNN(x_train_normalized, y_train, sample_normalized, k_value)

            img_name = os.path.basename(img_path)
            status = "✓" if true_label == predicted_label else "✗"
            print(f"Image: {img_name:<20} | True: {label_to_name[true_label]:<10} | "
                  f"Predicted: {label_to_name[predicted_label]:<10} {status}")

            if true_label == predicted_label:
                correct_predictions += 1
                class_correct[true_label] += 1

            total_predictions += 1
            class_total[true_label] += 1

    # ------Results-----
    print("\n" + "=" * 60)
    print("RESULTS")
    print("=" * 60)

    accuracy = (correct_predictions / total_predictions) * 100

    print(f"\nOverall Accuracy: {accuracy:.2f}%")
    print(f"Out of {total_predictions} test images, {correct_predictions} were correctly classified.")

    print("\nClass-wise Accuracy:")
    for label, name in label_to_name.items():
        if class_total[label] > 0:
            class_acc = (class_correct[label] / class_total[label]) * 100
            print(f"  {name.capitalize():<10}: {class_correct[label]}/{class_total[label]} "
                  f"({class_acc:.2f}%)")

    print("\n" + "=" * 60)

    return x_train_normalized, y_train, accuracy


# ------RUN THE PROGRAM------

if __name__ == "__main__":
    x_train, y_train, accuracy = main()
    print("\nProgram completed successfully!")
    print(f"Final Test Accuracy: {accuracy:.2f}%")


extract_features function defined.
KNN function defined.
STEP 1: TRAIN DATA LOADING
Processing training data
-> Processing camera (1) class: 40 images
-> Processing flamingo (2) class: 53 images
-> Processing pizza (3) class: 42 images

Training data prepared.
x_train shape: (135, 13)
y_train shape: (135,)

STEP 2: FEATURE NORMALIZATION
Normalization completed.
Train mean: [1679.22178729    6.07494052 1924.38913011] (first 3 features)
Train std: [647.03165169   0.82176383 511.63468292] (first 3 features)
Normalized data mean: -0.000000
Normalized data std: 0.999864

STEP 3: TESTING WITH KNN (k=5)

k value: 5
Testing started
------------------------------------------------------------

Testing camera images
Image: image_0046.jpg       | True: camera     | Predicted: flamingo   ✗
Image: image_0024.jpg       | True: camera     | Predicted: camera     ✓
Image: image_0040.jpg       | True: camera     | Predicted: camera     ✓
Image: image_0035.jpg       | True: camera     | Predicted: camer